# Loading a NeuroStore parquet Studyset

This notebook demonstrates loading a NiMARE `Studyset` from a directory of parquet files. The local data are a small slice of the NeuroStore nightly release stored as test resources.

In the future, the main use case for this format will be distributed downloads from [www.neurostore.org/api/neurostore-studyset-releases/](https://www.neurostore.org/api/neurostore-studyset-releases/). Those release archives are expected to contain the same table-oriented layout shown here.


## Parquet Studyset layout

A parquet-backed Studyset directory contains a `studyset.json` manifest plus canonical tables:

- `studies.parquet`: one row per study, with `study_id`, `name`, `description`, `authors`, and `publication`.
- `analyses.parquet`: one row per analysis, with full analysis `id`, `study_id`, `contrast_id`, and `name`.
- `coordinates.parquet`: coordinate rows keyed by `id`, `study_id`, and `contrast_id`, with `x`, `y`, `z`, `space`, and optional point metadata/value columns.
- `metadata.parquet`: one row per analysis with study/analysis descriptors and sample-size metadata when available.
- `annotations.parquet`: one row per analysis with annotation feature columns. All columns are loaded by default.
- `images.parquet`: image references keyed by analysis.
- `texts.parquet`: text fields keyed by analysis.

The `studyset.json` manifest records the studyset id/name, schema version, annotation ids, and table filenames.


In [ ]:
from pathlib import Path

import pandas as pd

from nimare.nimads import Studyset
from nimare.utils import get_resource_path

parquet_dir = Path(get_resource_path()) / "neurostore_parquet_studyset"
parquet_dir


In [ ]:
studyset = Studyset(parquet_dir)

print(studyset)
print(f"Studyset id: {studyset.id}")
print(f"Number of studies: {len(studyset.study_ids)}")
print(f"Number of analyses: {len(studyset.ids)}")
print(f"Materialized nested objects? {studyset.is_materialized}")


In [ ]:
table_shapes = {}
for table_file in sorted(parquet_dir.glob("*.parquet")):
    table = pd.read_parquet(table_file)
    table_shapes[table_file.name] = table.shape

table_shapes


In [ ]:
studyset.coordinates.head()


In [ ]:
annotation_columns = [
    column
    for column in studyset.annotations_df.columns
    if column not in {"id", "study_id", "contrast_id"}
]

print(f"Annotation feature columns: {len(annotation_columns)}")
studyset.annotations_df[["id"] + annotation_columns[:5]].head()


The object remains lazy after table loading. Accessing `studyset.studies` materializes nested `Study`, `Analysis`, and `Point` objects only when that object graph is needed. Most NiMARE workflows can operate directly on the table-backed views.
